# Proyecto: Predicción de Churn en Distribución de Streaming

## Contexto del Negocio
Este proyecto se desarrolla dentro de un modelo de emprendimiento enfocado en la **distribución y reventa de perfiles de plataformas de streaming** (MAX, Netflix, Crunchyroll, etc.).

### Objetivos
- **Variable Target (objetivo):** `churn` (0 = cliente activo, 1 = cliente que abandonó)
- **Técnica básica:** Regresión Logística
- **Técnica avanzada:** Random Forest (comparación de al menos 2 técnicas)

### Cronograma
| Semana | Actividad |
|--------|----------|
| 1 | Dataset, objetivos y cronograma |
| 2 | Preprocesamiento y análisis estadístico |
| 3 | Selección de variables y Regresión Logística |
| 4 | Random Forest y comparación de técnicas |
| 5 | Entrega final y documentación |

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
print('Librerías importadas correctamente')

## 2. Carga del Dataset

In [ ]:
df = pd.read_csv('dataset_streaming_limpio.csv')
print(f'Dimensiones del dataset: {df.shape}')
print(f'\nColumnas: {list(df.columns)}')
df.head(10)

## 3. Inspección Inicial del Dataset

In [ ]:
print('=== INFORMACIÓN DEL DATASET ===')
print(df.info())
print('\n=== ESTADÍSTICAS DESCRIPTIVAS ===')
df.describe()

In [ ]:
print('=== VALORES NULOS ===')
print(df.isnull().sum())
print(f'\n=== DISTRIBUCIÓN DEL TARGET (churn) ===')
print(df['churn'].value_counts())
print(f'\nTasa de Churn: {df["churn"].mean():.2%}')

## 4. Análisis Exploratorio de Datos (EDA)

### 4.1 Distribución de la Variable Target

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
df['churn'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'], ax=ax)
ax.set_title('Distribución de Churn')
ax.set_xlabel('Churn (0=Activo, 1=Abandonó)')
ax.set_ylabel('Cantidad')
ax.set_xticklabels(['Activo', 'Abandonó'], rotation=0)
plt.tight_layout()
plt.show()

### 4.2 Distribución por Plataforma

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
df['plataforma'].value_counts().plot(kind='bar', color='steelblue', ax=ax)
ax.set_title('Clientes por Plataforma')
ax.set_xlabel('Plataforma')
ax.set_ylabel('Cantidad')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.3 Churn por Duración del Plan

In [ ]:
churn_por_duracion = df.groupby('duracion_plan')['churn'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
churn_por_duracion.plot(kind='bar', color='coral', ax=ax)
ax.set_title('Tasa de Churn por Duración del Plan')
ax.set_xlabel('Duración del Plan')
ax.set_ylabel('Tasa de Churn')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 4.4 Churn por Plataforma

In [ ]:
churn_por_plat = df.groupby('plataforma')['churn'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
churn_por_plat.plot(kind='bar', color='teal', ax=ax)
ax.set_title('Tasa de Churn por Plataforma')
ax.set_xlabel('Plataforma')
ax.set_ylabel('Tasa de Churn')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Preprocesamiento de Datos

### 5.1 Codificación de Variables Categóricas

In [ ]:
# Codificar variables categóricas
le_plataforma = LabelEncoder()
le_tipo_plan = LabelEncoder()

df['plataforma_encoded'] = le_plataforma.fit_transform(df['plataforma'])
df['tipo_plan_encoded'] = le_tipo_plan.fit_transform(df['tipo_plan'])

print('Mapeo de Plataforma:')
for i, clase in enumerate(le_plataforma.classes_):
    print(f'  {clase} -> {i}')

print('\nMapeo de Tipo de Plan:')
for i, clase in enumerate(le_tipo_plan.classes_):
    print(f'  {clase} -> {i}')

### 5.2 Selección de Variables Predictoras

In [ ]:
# Variables predictoras (features)
features = ['duracion_meses', 'plataforma_encoded', 'tipo_plan_encoded',
             'mes_inicio', 'tiene_fecha_inicio']

X = df[features].copy()
y = df['churn'].copy()

print(f'Features seleccionadas: {features}')
print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')
print(f'\nPrimeras filas de X:')
X.head()

### 5.3 Matriz de Correlación

In [ ]:
# Agregar target para la correlación
df_corr = X.copy()
df_corr['churn'] = y

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_corr.corr(), annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f')
ax.set_title('Matriz de Correlación')
plt.tight_layout()
plt.show()

### 5.4 División Train/Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Conjunto de entrenamiento: {X_train.shape[0]} muestras')
print(f'Conjunto de prueba: {X_test.shape[0]} muestras')
print(f'\nDistribución de churn en entrenamiento:')
print(y_train.value_counts())
print(f'\nDistribución de churn en prueba:')
print(y_test.value_counts())

### 5.5 Escalamiento de Features

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Datos escalados correctamente')
print(f'Media de X_train_scaled: {X_train_scaled.mean(axis=0).round(4)}')
print(f'Std de X_train_scaled: {X_train_scaled.std(axis=0).round(4)}')

## 6. Técnica 1: Regresión Logística

### 6.1 Entrenamiento del Modelo

In [ ]:
# Crear y entrenar el modelo de Regresión Logística
lr = LogisticRegression(penalty='l2', C=1.0, random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)

print('Modelo de Regresión Logística entrenado')
print(f'\nIntercepto (w0): {lr.intercept_[0]:.4f}')
print(f'\nCoeficientes:')
for feat, coef in zip(features, lr.coef_[0]):
    print(f'  {feat}: {coef:.4f}')

### 6.2 Predicciones

In [ ]:
# Predicciones
y_pred_lr = lr.predict(X_test_scaled)
y_probs_lr = lr.predict_proba(X_test_scaled)[:, 1]

print('Primeras 10 predicciones vs valores reales:')
for i in range(min(10, len(y_test))):
    real = y_test.iloc[i]
    pred = y_pred_lr[i]
    prob = y_probs_lr[i]
    print(f'  Real: {real} | Predicción: {pred} | P(churn): {prob:.4f}')

### 6.3 Evaluación - Regresión Logística

In [ ]:
# Métricas
print('=== REPORTE DE CLASIFICACIÓN - Regresión Logística ===')
print(classification_report(y_test, y_pred_lr, target_names=['Activo', 'Churn']))

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Accuracy: {acc_lr:.4f}')

In [ ]:
# Matriz de Confusión
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Activo', 'Churn'], yticklabels=['Activo', 'Churn'])
ax.set_title('Matriz de Confusión - Regresión Logística')
ax.set_xlabel('Predicción')
ax.set_ylabel('Valor Real')
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
auc_lr = roc_auc_score(y_test, y_probs_lr)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_probs_lr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_lr, tpr_lr, color='red', label=f'Reg. Logística (AUC = {auc_lr:.2f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Curva ROC - Regresión Logística')
ax.legend()
plt.tight_layout()
plt.show()
print(f'ROC-AUC Score: {auc_lr:.4f}')

## 7. Técnica 2: Random Forest

Un **Random Forest** entrena varios árboles de decisión. Cada árbol se entrena con una muestra aleatoria de los datos y con subconjuntos aleatorios de variables. Finalmente, el bosque decide por **votación mayoritaria**. Esto produce modelos más robustos que un solo árbol de decisión.

Seguimos la misma metodología vista en clase:
1. Entrenamiento del modelo
2. Evaluación con tabla train/test (accuracy, precision, recall, F1)
3. Reporte de clasificación completo
4. Matriz de confusión con `ConfusionMatrixDisplay`
5. Importancia de variables
6. Exploración interna: votación de los árboles para un caso
7. Estadísticas de profundidad y hojas del bosque
8. Comparación RF vs. mejor árbol del bosque vs. árbol independiente (GridSearchCV)
9. Experimentación con hiperparámetros

### 7.1 Entrenamiento del Modelo

In [ ]:
# Entrenamiento del Random Forest
# n_estimators=200 árboles, class_weight='balanced' para manejar desbalance de clases
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced'
)

rf.fit(X_train_scaled, y_train)

y_train_pred = rf.predict(X_train_scaled)
y_test_pred  = rf.predict(X_test_scaled)

print('Modelo Random Forest entrenado exitosamente')
print(f'Número de árboles: {rf.n_estimators}')

### 7.2 Evaluación del Modelo: Train vs. Test

Comparamos el rendimiento en entrenamiento y en prueba para detectar posible **sobreajuste**: si el accuracy de train es mucho mayor que el de test, el modelo memorizó los datos en lugar de aprender patrones generalizables.

In [ ]:
def evaluar_modelo(nombre, y_real, y_pred):
    """Calcula métricas principales para un modelo de clasificación binaria."""
    return pd.Series({
        'modelo': nombre,
        'accuracy':  accuracy_score(y_real, y_pred),
        'precision': precision_score(y_real, y_pred, zero_division=0),
        'recall':    recall_score(y_real, y_pred, zero_division=0),
        'f1_score':  f1_score(y_real, y_pred, zero_division=0)
    })

resultados_rf = pd.DataFrame([
    evaluar_modelo('Random Forest - train', y_train, y_train_pred),
    evaluar_modelo('Random Forest - test',  y_test,  y_test_pred)
])

resultados_rf.set_index('modelo').round(3)

### 7.3 Reporte de Clasificación

- **Precision:** de los clientes predichos como churn, ¿cuántos realmente abandonaron?
- **Recall:** de los clientes que realmente abandonaron, ¿cuántos detectó el modelo?
- **F1-score:** media armónica entre precision y recall.

In [ ]:
print('Reporte de clasificación en el conjunto de prueba:')
print(classification_report(y_test, y_test_pred, target_names=['Activo (0)', 'Churn (1)']))

### 7.4 Matriz de Confusión

La matriz de confusión muestra en qué casos acierta y en cuáles se equivoca el modelo:
- **Verdaderos negativos (TN):** clientes activos correctamente clasificados
- **Falsos positivos (FP):** clientes activos clasificados como churn
- **Falsos negativos (FN):** clientes churn que el modelo no detectó ⚠️ (los más costosos para el negocio)
- **Verdaderos positivos (TP):** clientes churn correctamente detectados

In [ ]:
cm_rf = confusion_matrix(y_test, y_test_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_rf,
    display_labels=['Activo', 'Churn']
)

disp.plot(cmap='Blues', values_format='d')
plt.title('Matriz de Confusión - Random Forest')
plt.tight_layout()
plt.show()

### 7.5 Importancia de Variables

Random Forest permite estimar qué variables fueron más útiles para dividir los datos. La importancia se calcula a partir de la **reducción promedio de impureza** (Gini) a través de todos los árboles.

In [ ]:
importancias = pd.DataFrame({
    'variable':    features,
    'importancia': rf.feature_importances_
}).sort_values(by='importancia', ascending=False)

print('Importancia de variables:')
importancias.round(4)

In [ ]:
plt.figure(figsize=(7, 5))
plt.barh(importancias['variable'], importancias['importancia'], color='forestgreen')
plt.gca().invert_yaxis()
plt.xlabel('Importancia')
plt.title('Importancia de Variables en Random Forest')
plt.tight_layout()
plt.show()

### 7.6 Exploración Interna: ¿Cómo Votan los Árboles?

Cada árbol produce una predicción individual. El Random Forest toma la **clase más votada**. Analicemos un cliente específico del conjunto de prueba para ver cómo se distribuyen los votos.

In [ ]:
print(f'Número de árboles en el bosque: {len(rf.estimators_)}')
print(type(rf.estimators_[0]))

In [ ]:
# Analizar el cliente en el índice 5 del conjunto de prueba
idx0 = 5

new_x       = X_test.iloc[[idx0]]          # conservar formato DataFrame
new_x_sc    = X_test_scaled[idx0:idx0+1]   # versión escalada
real_label  = y_test.iloc[idx0]
rf_label    = rf.predict(new_x_sc)[0]

print(f'Índice analizado dentro de X_test: {idx0}')
print(f'Etiqueta real:                     {real_label}  (0=Activo, 1=Churn)')
print(f'Predicción del Random Forest:      {rf_label}')

# Predicción de cada árbol individual
predicciones_arboles = np.array([
    arbol.predict(new_x_sc)[0] for arbol in rf.estimators_
])

votos_0 = int(np.sum(predicciones_arboles == 0))
votos_1 = int(np.sum(predicciones_arboles == 1))

print(f'\nVotos para Activo (0): {votos_0}')
print(f'Votos para Churn  (1): {votos_1}')
print(f'Primeras 20 predicciones individuales: {predicciones_arboles[:20]}')

In [ ]:
plt.figure(figsize=(5, 4))
plt.bar(['Activo (0)', 'Churn (1)'], [votos_0, votos_1], color=['steelblue', 'salmon'])
plt.ylabel('Número de votos')
plt.title(f'Votación de los árboles — cliente índice {idx0}')
plt.tight_layout()
plt.show()

### 7.7 Exploración de un Árbol Individual

Podemos inspeccionar cualquier árbol del bosque para ver su profundidad, número de hojas y sus reglas de decisión (hasta cierta profundidad para no imprimir un árbol gigante).

In [ ]:
arbol_individual = rf.estimators_[1]

print(f'Profundidad del árbol:       {arbol_individual.get_depth()}')
print(f'Número de hojas del árbol:   {arbol_individual.get_n_leaves()}')
print()
print('Primeras 3 capas de reglas del árbol individual:')
reglas = export_text(arbol_individual, feature_names=features, max_depth=3)
print(reglas)

### 7.8 Estadísticas de Todos los Árboles del Bosque

No todos los árboles tienen la misma profundidad ni el mismo número de hojas. Veamos la distribución de estos valores en los 200 árboles.

In [ ]:
profundidades = [arbol.get_depth()    for arbol in rf.estimators_]
hojas         = [arbol.get_n_leaves() for arbol in rf.estimators_]

resumen_arboles = pd.DataFrame({'profundidad': profundidades, 'numero_hojas': hojas})
resumen_arboles.describe().round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(profundidades, bins=15, color='steelblue')
axes[0].set_xlabel('Profundidad')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de profundidades de los árboles')

axes[1].hist(hojas, bins=15, color='teal')
axes[1].set_xlabel('Número de hojas')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución del número de hojas de los árboles')

plt.tight_layout()
plt.show()

### 7.9 Comparación: Random Forest vs. Árboles Individuales

Comparamos tres modelos:
1. **Random Forest completo** (200 árboles)
2. **Mejor árbol individual dentro del bosque** (el que tiene mejor accuracy en test)
3. **Árbol de decisión independiente** optimizado con `GridSearchCV`

In [ ]:
# Accuracy de cada árbol individual del bosque en el conjunto de prueba
accuracies_arboles = []

for arbol in rf.estimators_:
    pred_i = arbol.predict(X_test_scaled)
    acc_i  = accuracy_score(y_test, pred_i)
    accuracies_arboles.append(acc_i)

mejor_idx        = int(np.argmax(accuracies_arboles))
mejor_arbol_bosque = rf.estimators_[mejor_idx]

print(f'Mejor árbol dentro del bosque: índice {mejor_idx}')
print(f'Accuracy del mejor árbol individual: {accuracies_arboles[mejor_idx]:.3f}')
print(f'Profundidad: {mejor_arbol_bosque.get_depth()}')
print(f'Número de hojas: {mejor_arbol_bosque.get_n_leaves()}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(accuracies_arboles, bins=15, color='steelblue', alpha=0.7)
plt.axvline(accuracy_score(y_test, y_test_pred), linestyle='--', color='red',
            label=f'Random Forest completo ({accuracy_score(y_test, y_test_pred):.3f})')
plt.xlabel('Accuracy en test')
plt.ylabel('Número de árboles')
plt.title('Accuracy de los árboles individuales vs. el bosque completo')
plt.legend()
plt.tight_layout()
plt.show()

#### Árbol de Decisión Independiente con GridSearchCV

Entrenamos un árbol de decisión independiente y usamos **búsqueda en cuadrícula** para encontrar la mejor combinación de hiperparámetros (`max_depth`, `min_samples_leaf`).

In [ ]:
param_grid = {
    'max_depth':        [2, 3, 4, 5, 6, None],
    'min_samples_leaf': [1, 5, 10, 20]
}

dt_base = DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced')

grid_dt = GridSearchCV(
    estimator=dt_base,
    param_grid=param_grid,
    scoring='f1',
    cv=5,
    n_jobs=-1
)

grid_dt.fit(X_train_scaled, y_train)

mejor_dt         = grid_dt.best_estimator_
y_test_pred_dt   = mejor_dt.predict(X_test_scaled)

print('Mejores hiperparámetros del árbol independiente:')
print(grid_dt.best_params_)
print(f'Profundidad: {mejor_dt.get_depth()}')
print(f'Número de hojas: {mejor_dt.get_n_leaves()}')

In [ ]:
# Tabla comparativa de los tres modelos
pred_mejor_arbol_bosque = mejor_arbol_bosque.predict(X_test_scaled)

comparacion_arboles = pd.DataFrame([
    evaluar_modelo('Random Forest completo (200 árboles)', y_test, y_test_pred),
    evaluar_modelo('Mejor árbol individual del bosque',    y_test, pred_mejor_arbol_bosque),
    evaluar_modelo('Árbol de decisión independiente',      y_test, y_test_pred_dt)
])

print('=== COMPARACIÓN INTERNA DE ÁRBOLES ===')
comparacion_arboles.set_index('modelo').round(3)

### 7.10 Experimentación con Hiperparámetros

Modificamos los hiperparámetros del Random Forest para observar cómo cambian las métricas. Basado en la actividad de la práctica de clase:
- `n_estimators`: número de árboles
- `max_depth`: profundidad máxima de cada árbol
- `min_samples_leaf`: mínimo de muestras en hojas (regularización)
- `class_weight='balanced'`: compensa el desbalance de clases

In [ ]:
rf_experimento = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight='balanced'
)

rf_experimento.fit(X_train_scaled, y_train)

pred_train_exp = rf_experimento.predict(X_train_scaled)
pred_test_exp  = rf_experimento.predict(X_test_scaled)

pd.DataFrame([
    evaluar_modelo('RF Experimento - train', y_train, pred_train_exp),
    evaluar_modelo('RF Experimento - test',  y_test,  pred_test_exp)
]).set_index('modelo').round(3)

## 8. Comparación de Modelos

In [ ]:
# Curvas ROC comparativas: Regresión Logística vs. Random Forest
auc_rf  = roc_auc_score(y_test, rf.predict_proba(X_test_scaled)[:, 1])
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf.predict_proba(X_test_scaled)[:, 1])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr_lr, tpr_lr, color='red',   label=f'Reg. Logística (AUC = {auc_lr:.2f})')
ax.plot(fpr_rf, tpr_rf, color='green', label=f'Random Forest  (AUC = {auc_rf:.2f})')
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos')
ax.set_title('Comparación de Curvas ROC')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Tabla comparativa final entre técnicas
acc_rf  = accuracy_score(y_test, y_test_pred)

comparacion_final = pd.DataFrame({
    'Modelo':   ['Regresión Logística', 'Random Forest'],
    'Accuracy': [acc_lr, acc_rf],
    'ROC-AUC':  [auc_lr, auc_rf]
})
print('=== COMPARACIÓN FINAL DE MODELOS ===')
print(comparacion_final.to_string(index=False))

mejor = 'Regresión Logística' if auc_lr > auc_rf else 'Random Forest'
print(f'\nMejor modelo según ROC-AUC: {mejor}')

## 9. Conclusiones

### Hallazgos Principales
1. Se procesaron datos reales del negocio de distribución de streaming (~245 registros)
2. La tasa de churn observada es de aproximadamente 26%, indicando que 1 de cada 4 clientes abandona
3. Los planes de corta duración (1 mes) presentan tasas de churn más altas
4. Se implementaron y compararon 2 técnicas de clasificación: Regresión Logística y Random Forest

### Aplicación al Negocio
- El modelo permite **anticipar bajas** antes de que venzan las suscripciones
- Permite diseñar **campañas de retención** dirigidas a clientes con alta probabilidad de churn
- La variable `duracion_meses` es un predictor importante del churn